




















RetailPulse 360


Notebook 09 — Inventory Redistribution Engine

Goal: This is the actual decision-intelligence deliverable — not "here's a forecast" or
"here's current stock," but "move X units of this product from Store A to Store B, and
\here's exactly why." Combines Notebook 06's inventory snapshots, Notebook 08's demand
forecasts, and real store geography (from Notebook 02) to produce prioritized, explainable
transfer recommendations. Candidate transfers are allowed anywhere in Pakistan — real
distance is factored into the priority score rather than restricting to same-region only,
since a hard boundary doesn't reflect real logistics as accurately as actual distance does.

Input: inventory_turnover_summary.csv, demand_forecast_model.pkl, sales.csv, stores.csv,
       cities_with_coords.csv, skus.csv, products.csv
Output: redistribution_recommendations.csv

In [3]:
# 1. IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import joblib

print("Libraries imported successfully.")

Libraries imported successfully.


In [4]:
# 2. LOAD ALL INPUTS
# ============================================================

BASE_PATH = "/kaggle/input/datasets/hamaz911/notebook-9-dataset/"

inventory = pd.read_csv(BASE_PATH + "inventory_turnover_summary.csv")
stores = pd.read_csv(BASE_PATH + "stores.csv")
cities = pd.read_csv(BASE_PATH + "cities_with_coords.csv")
skus = pd.read_csv("/kaggle/input/datasets/hamaz911/skusss/skus.csv")
products = pd.read_csv(BASE_PATH + "products.csv")
sales = pd.read_csv(BASE_PATH + "sales.csv", parse_dates=["date"])
model = joblib.load("/kaggle/input/datasets/hamaz911/forecasting-model/demand_forecast_model.pkl")

print("Loaded:")
for name, df in [("inventory", inventory), ("stores", stores), ("cities", cities),
                  ("skus", skus), ("products", products), ("sales", sales)]:
    print(f"  {name}: {df.shape}")
print("  model:", type(model).__name__, "loaded successfully")

Loaded:
  inventory: (13678, 7)
  stores: (191, 18)
  cities: (94, 7)
  skus: (3296, 13)
  products: (72, 7)
  sales: (2459464, 4)
  model: LGBMRegressor loaded successfully


In [5]:
# 3. DATA QUALITY — VALIDATE INPUTS FOR REDISTRIBUTION LOGIC
# ============================================================

print("Inventory status breakdown:")
print(inventory["stock_status"].value_counts())

print("\nMissing values in inventory:")
missing = inventory.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None")

print("\nCities missing lat/lng (needed for distance calc):", cities[["lat","lng"]].isna().any(axis=1).sum())

print("\nReferential integrity:")
print("  inventory -> stores orphans:", (~inventory["store_id"].isin(stores["store_id"])).sum())
print("  inventory -> products orphans:", (~inventory["product_id"].isin(products["product_id"])).sum())
print("  stores -> cities orphans:", (~stores["city"].isin(cities["city"])).sum())

Inventory status breakdown:
stock_status
Healthy               5153
Low                   4890
Critical (Reorder)    3469
Overstock              157
Fully Dead               9
Name: count, dtype: int64

Missing values in inventory:
active_daily_velocity    9
days_of_supply           9
dtype: int64

Cities missing lat/lng (needed for distance calc): 0

Referential integrity:
  inventory -> stores orphans: 0
  inventory -> products orphans: 0
  stores -> cities orphans: 1


In [6]:
# 3b. INVESTIGATE — WHICH STORE'S CITY DOESN'T MATCH?
# ============================================================

orphan_store = stores[~stores["city"].isin(cities["city"])]
print(orphan_store[["store_id", "city", "region", "channel"]])

     store_id               city    region channel
190  STY-ECOM  National (Online)  National  Online


In [7]:
# 4. BUILD NEED AND SUPPLY POOLS (PHYSICAL STORES ONLY)
# ============================================================
# STY-ECOM has no physical location -- can't be a source or destination
# for a physical inventory transfer.

physical_inventory = inventory[inventory["store_id"] != "STY-ECOM"].copy()

need_pool = physical_inventory[physical_inventory["stock_status"].isin(["Critical (Reorder)", "Low"])].copy()
supply_pool = physical_inventory[physical_inventory["stock_status"].isin(["Overstock", "Fully Dead"])].copy()

print("Need pool (understocked):", len(need_pool))
print("Supply pool (overstocked/dead):", len(supply_pool))

print("\nHow many products appear in BOTH pools (i.e., a real match is even possible)?")
common_products = set(need_pool["product_id"]) & set(supply_pool["product_id"])
print("Products with both a need and a supply somewhere:", len(common_products))
print("Total distinct products:", products["product_id"].nunique())

Need pool (understocked): 8359
Supply pool (overstocked/dead): 166

How many products appear in BOTH pools (i.e., a real match is even possible)?
Products with both a need and a supply somewhere: 55
Total distinct products: 72


In [9]:
# 5. HAVERSINE DISTANCE + ROAD-DISTANCE ADJUSTMENT
# ============================================================
# Straight-line (great-circle) distance is always shorter than actual
# road distance, since roads curve around terrain. Sourced adjustment:
# Lahore-Karachi straight-line ~1,030km vs real road/rail ~1,214-1,234km
# -> roughly 1.17-1.2x. We apply 1.17x uniformly as a reasonable,
# documented approximation for real trucking distance.

ROAD_DISTANCE_FACTOR = 1.17

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def road_distance_km(lat1, lon1, lat2, lon2):
    return haversine_km(lat1, lon1, lat2, lon2) * ROAD_DISTANCE_FACTOR

# Re-verify with the adjustment applied
lahore = cities[cities["city"] == "Lahore"].iloc[0]
karachi = cities[cities["city"] == "Karachi"].iloc[0]
test_dist = road_distance_km(lahore["lat"], lahore["lng"], karachi["lat"], karachi["lng"])
print(f"Lahore -> Karachi estimated road distance: {test_dist:.0f} km (real road/rail: ~1214-1234 km)")

Lahore -> Karachi estimated road distance: 1209 km (real road/rail: ~1214-1234 km)


In [15]:
# 6. BUILD MATCHING ALGORITHM (SAFE TO RE-RUN)
# ============================================================

store_coords = stores.merge(cities[["city", "lat", "lng"]], on="city", how="left")

# Drop lat/lng if already present (makes this cell safely re-runnable)
need_pool = need_pool.drop(columns=["lat", "lng"], errors="ignore")
supply_pool = supply_pool.drop(columns=["lat", "lng"], errors="ignore")

need_pool = need_pool.merge(store_coords[["store_id", "lat", "lng"]], on="store_id", how="left")
supply_pool = supply_pool.merge(store_coords[["store_id", "lat", "lng"]], on="store_id", how="left")

TARGET_DAYS_BY_SIZE = {"Large": 28, "Medium": 21, "Small": 14}
supply_pool = supply_pool.drop(columns=["store_size"], errors="ignore")
supply_pool = supply_pool.merge(stores[["store_id", "store_size"]], on="store_id", how="left")
supply_pool["target_days"] = supply_pool["store_size"].map(TARGET_DAYS_BY_SIZE)

supply_pool["active_daily_velocity"] = supply_pool["active_daily_velocity"].fillna(0)
supply_pool["active_stock"] = supply_pool["active_stock"].fillna(0)

supply_pool["own_target_stock"] = supply_pool["active_daily_velocity"] * supply_pool["target_days"]
supply_pool["transferable_surplus"] = np.maximum(
    0, supply_pool["active_stock"] - supply_pool["own_target_stock"]
).round().astype(int)

supply_pool.loc[supply_pool["active_daily_velocity"] == 0, "transferable_surplus"] = supply_pool["dead_stock_units"].astype(int)

print("Supply pool transferable surplus stats:")
print(supply_pool["transferable_surplus"].describe())
print("\nSupply stores with actual surplus to give (>0 units):",
      (supply_pool["transferable_surplus"] > 0).sum(), "out of", len(supply_pool))

Supply pool transferable surplus stats:
count    166.000000
mean       2.578313
std        1.979412
min        1.000000
25%        1.000000
50%        2.000000
75%        3.000000
max       10.000000
Name: transferable_surplus, dtype: float64

Supply stores with actual surplus to give (>0 units): 166 out of 166


In [16]:
# 7. GENERATE CANDIDATE TRANSFER PAIRS
# ============================================================
# Cross-join need x supply WITHIN each product (not across products --
# you can't fix a Formal shortage with Casual stock), then score.

candidates = []

for product_id in common_products:
    needs = need_pool[need_pool["product_id"] == product_id]
    supplies = supply_pool[supply_pool["product_id"] == product_id]

    for _, need_row in needs.iterrows():
        for _, supply_row in supplies.iterrows():
            if need_row["store_id"] == supply_row["store_id"]:
                continue  # can't transfer a store's stock to itself

            distance = road_distance_km(
                need_row["lat"], need_row["lng"],
                supply_row["lat"], supply_row["lng"]
            )

            # How many units does the need actually require? Enough to
            # close the gap to a healthy 21-day supply, capped by what's
            # actually available at the source.
            units_needed = max(0, round(need_row["active_daily_velocity"] * 21 - need_row["active_stock"]))
            transfer_qty = min(units_needed, supply_row["transferable_surplus"])

            if transfer_qty <= 0:
                continue

            # Expected benefit: extra days of supply this transfer buys
            # the destination store (avoid divide-by-zero for dead-velocity needs)
            days_gained = (transfer_qty / need_row["active_daily_velocity"]
                           if need_row["active_daily_velocity"] > 0 else 0)

            candidates.append({
                "product_id": product_id,
                "from_store": supply_row["store_id"],
                "to_store": need_row["store_id"],
                "transfer_qty": int(transfer_qty),
                "distance_km": round(distance, 1),
                "days_gained": round(days_gained, 1),
                "to_store_status": need_row["stock_status"],
                "from_store_status": supply_row["stock_status"],
            })

candidates_df = pd.DataFrame(candidates)
print("Total candidate transfer pairs generated:", len(candidates_df))
print("\nSample:")
print(candidates_df.head(10))

Total candidate transfer pairs generated: 13945

Sample:
  product_id from_store  to_store  transfer_qty  distance_km  days_gained  \
0  PROD-0008   STY-0008  STY-0002             1          0.0         17.2   
1  PROD-0008   STY-0076  STY-0002             1         80.1         17.2   
2  PROD-0008   STY-0093  STY-0002             1        796.3         17.2   
3  PROD-0008   STY-0139  STY-0002             1        965.6         17.2   
4  PROD-0008   STY-0155  STY-0002             1        568.7         17.2   
5  PROD-0008   STY-0157  STY-0002             1        242.7         17.2   
6  PROD-0008   STY-0177  STY-0002             1        731.2         17.2   
7  PROD-0008   STY-0008  STY-0003             1          0.0          4.0   
8  PROD-0008   STY-0076  STY-0003             1         80.1          4.0   
9  PROD-0008   STY-0093  STY-0003             1        796.3          4.0   

      to_store_status from_store_status  
0  Critical (Reorder)         Overstock  
1  Critical

In [17]:
# 8. PRIORITY SCORE + GREEDY ALLOCATION
# ============================================================
# A candidate pair alone doesn't guarantee the transfer is actually
# possible once OTHER destinations compete for the same limited source
# surplus. Greedy allocation: rank by value, then allocate stock in that
# order, depleting each source as we go -- so no source gets
# over-committed across multiple recommendations.

# Priority score: days of supply gained per 100km of distance
# (rewards high benefit, penalizes distance smoothly, no divide-by-zero)
candidates_df["priority_score"] = candidates_df["days_gained"] / (candidates_df["distance_km"] / 100 + 1)

candidates_sorted = candidates_df.sort_values("priority_score", ascending=False).reset_index(drop=True)

remaining_supply = supply_pool.set_index(["store_id", "product_id"])["transferable_surplus"].to_dict()
remaining_need = need_pool.copy()
remaining_need["units_still_needed"] = np.maximum(
    0, (remaining_need["active_daily_velocity"] * 21 - remaining_need["active_stock"]).round()
).astype(int)
remaining_need_dict = remaining_need.set_index(["store_id", "product_id"])["units_still_needed"].to_dict()

final_recommendations = []
for _, row in candidates_sorted.iterrows():
    supply_key = (row["from_store"], row["product_id"])
    need_key = (row["to_store"], row["product_id"])

    available = remaining_supply.get(supply_key, 0)
    still_needed = remaining_need_dict.get(need_key, 0)
    actual_qty = min(available, still_needed)

    if actual_qty <= 0:
        continue

    remaining_supply[supply_key] -= actual_qty
    remaining_need_dict[need_key] -= actual_qty

    final_recommendations.append({
        "product_id": row["product_id"],
        "from_store": row["from_store"],
        "to_store": row["to_store"],
        "transfer_qty": int(actual_qty),
        "distance_km": row["distance_km"],
        "days_gained": round(actual_qty / (remaining_need.set_index(["store_id","product_id"]).loc[need_key, "active_daily_velocity"] or 1), 1),
        "priority_score": round(row["priority_score"], 2),
        "to_store_status": row["to_store_status"],
        "from_store_status": row["from_store_status"],
    })

recommendations_df = pd.DataFrame(final_recommendations)
print("Final allocated recommendations:", len(recommendations_df))
print("Total units recommended for transfer:", recommendations_df["transfer_qty"].sum())
print("\nHow many of the 8,359 need-situations got at least partially addressed?",
      recommendations_df["to_store"].nunique() * recommendations_df.groupby("to_store")["product_id"].nunique().mean())
print(recommendations_df.head(10))

Final allocated recommendations: 350
Total units recommended for transfer: 428

How many of the 8,359 need-situations got at least partially addressed? 350.0
  product_id from_store  to_store  transfer_qty  distance_km  days_gained  \
0  PROD-0013   STY-0056  STY-0054             1          0.0         41.5   
1  PROD-0033   STY-0119  STY-0120             1          0.0         41.5   
2  PROD-0036   STY-0015  STY-0012             1          0.0         39.7   
3  PROD-0047   STY-0053  STY-0054             1          0.0         39.7   
4  PROD-0005   STY-0077  STY-0074             1          0.0         38.0   
5  PROD-0028   STY-0020  STY-0035             1          0.0         38.0   
6  PROD-0022   STY-0053  STY-0054             1          0.0         36.5   
7  PROD-0010   STY-0011  STY-0002             1          0.0         35.1   
8  PROD-0025   STY-0115  STY-0116             1          0.0         33.8   
9  PROD-0009   STY-0122  STY-0074             1         23.3         41.

In [18]:
# 8b. FIX — CORRECT METRIC FOR NEED-SITUATIONS ADDRESSED
# ============================================================
# My earlier line multiplied two unrelated counts together -- meaningless.
# The real number: how many distinct (store, product) shortage situations
# got at least one recommendation.

addressed = recommendations_df.groupby(["to_store", "product_id"]).ngroups
print(f"Need-situations addressed: {addressed} out of {len(need_pool):,} total Critical/Low situations")
print(f"That's {addressed/len(need_pool)*100:.1f}% of all shortage situations")

Need-situations addressed: 350 out of 8,359 total Critical/Low situations
That's 4.2% of all shortage situations


In [19]:
# 9. GENERATE EXPLAINABLE REASONING TEXT
# ============================================================

recommendations_df = recommendations_df.merge(
    stores[["store_id", "city"]].rename(columns={"store_id": "from_store", "city": "from_city"}),
    on="from_store", how="left"
)
recommendations_df = recommendations_df.merge(
    stores[["store_id", "city"]].rename(columns={"store_id": "to_store", "city": "to_city"}),
    on="to_store", how="left"
)
recommendations_df = recommendations_df.merge(
    products[["product_id", "style_name"]], on="product_id", how="left"
)

def build_reasoning(row):
    return (
        f"Transfer {row['transfer_qty']} unit(s) of '{row['style_name']}' from "
        f"{row['from_city']} ({row['from_store']}, currently {row['from_store_status']}) "
        f"to {row['to_city']} ({row['to_store']}, currently {row['to_store_status']}). "
        f"Distance: {row['distance_km']:.0f} km. "
        f"Expected impact: gains approximately {row['days_gained']:.1f} extra days of stock coverage "
        f"at the destination store."
    )

recommendations_df["reasoning"] = recommendations_df.apply(build_reasoning, axis=1)

print("Sample reasoning (top 3 by priority):")
for _, row in recommendations_df.head(3).iterrows():
    print("-", row["reasoning"])

Sample reasoning (top 3 by priority):
- Transfer 1 unit(s) of 'Men's Formal Style 5' from Faisalabad (STY-0056, currently Overstock) to Faisalabad (STY-0054, currently Critical (Reorder)). Distance: 0 km. Expected impact: gains approximately 41.5 extra days of stock coverage at the destination store.
- Transfer 1 unit(s) of 'Women's Formal Style 1' from Okara (STY-0119, currently Overstock) to Okara (STY-0120, currently Critical (Reorder)). Distance: 0 km. Expected impact: gains approximately 41.5 extra days of stock coverage at the destination store.
- Transfer 1 unit(s) of 'Women's Formal Style 4' from Lahore (STY-0015, currently Overstock) to Lahore (STY-0012, currently Critical (Reorder)). Distance: 0 km. Expected impact: gains approximately 39.7 extra days of stock coverage at the destination store.


In [20]:
# 10. SAVE OUTPUTS
# ============================================================

output_cols = ["product_id", "style_name", "from_store", "from_city", "from_store_status",
               "to_store", "to_city", "to_store_status", "transfer_qty", "distance_km",
               "days_gained", "priority_score", "reasoning"]

recommendations_df[output_cols].to_csv("redistribution_recommendations.csv", index=False)
print("Saved redistribution_recommendations.csv —", recommendations_df.shape[0], "rows")

Saved redistribution_recommendations.csv — 350 rows


In [21]:
# 11. NOTEBOOK SUMMARY
# ============================================================

print("NOTEBOOK 09 SUMMARY — INVENTORY REDISTRIBUTION ENGINE")
print(f"Understocked situations (Critical/Low): {len(need_pool):,}")
print(f"Overstocked/dead situations available as supply: {len(supply_pool):,}")
print(f"Products with both a need and a supply somewhere: {len(common_products)} of {products['product_id'].nunique()}")
print(f"Total candidate transfer pairs evaluated: {len(candidates_df):,}")
print(f"Final recommended transfers (after allocation): {len(recommendations_df):,}")
print(f"Total units recommended for transfer: {recommendations_df['transfer_qty'].sum():,}")
print(f"Need-situations meaningfully addressed: {addressed} of {len(need_pool):,} ({addressed/len(need_pool)*100:.1f}%)")
print()
print("Honest limitation: the vast majority of shortage situations (95.8%) have")
print("NO surplus stock anywhere in the network to draw from -- redistribution")
print("recovers real value from existing dead/excess stock, but is not a")
print("substitute for ordering more inventory from the supplier.")
print()
print("Output: redistribution_recommendations.csv")
print("\n\u2713 Notebook 09 completed successfully.")

NOTEBOOK 09 SUMMARY — INVENTORY REDISTRIBUTION ENGINE
Understocked situations (Critical/Low): 8,359
Overstocked/dead situations available as supply: 166
Products with both a need and a supply somewhere: 55 of 72
Total candidate transfer pairs evaluated: 13,945
Final recommended transfers (after allocation): 350
Total units recommended for transfer: 428
Need-situations meaningfully addressed: 350 of 8,359 (4.2%)

Honest limitation: the vast majority of shortage situations (95.8%) have
NO surplus stock anywhere in the network to draw from -- redistribution
recovers real value from existing dead/excess stock, but is not a
substitute for ordering more inventory from the supplier.

Output: redistribution_recommendations.csv

✓ Notebook 09 completed successfully.
